In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

## Data Loading and Exploratory Data Analysis

In [ ]:
df = pd.read_csv('data/basketball_shooting_dataset.csv')
df.head()

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    display(df.groupby('Shot_Result').describe().T)

In [ ]:
numeric_cols = [col for col in df.select_dtypes(include='number').columns if col != 'Shot_Result']

df_std = df[['Shot_Result'] + numeric_cols].copy()
df_std[numeric_cols] = (df_std[numeric_cols] - df_std[numeric_cols].mean()) / df_std[numeric_cols].std()

n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    sns.boxplot(x='Shot_Result', y=col, data=df_std, ax=axes[idx], showfliers=True)
    axes[idx].axhline(0, color='grey', linestyle='--', linewidth=0.8, alpha=0.7)
    axes[idx].set_ylim(-3, 3)
    axes[idx].set_title(f'{col} (Standardized)')
    axes[idx].set_xlabel('Shot Made (0 = Miss, 1 = Make)')
    axes[idx].set_ylabel('Z-score')

for j in range(idx + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
candidates = ['Age', 'Height_cm', 'Distance_from_Basket_m', 'Shot_Angle_deg', 'Accelerometer_Y_m_s2', 'Gyroscope_Y_deg_s', 'Shot_Speed_m_s', 'Release_Angle_deg']

sns.heatmap(df[candidates].corr(), annot=True, cmap='coolwarm')

In [ ]:
# Compute sensor vector magnitudes once for downstream modeling
accel_axes = ['Accelerometer_X_m_s2', 'Accelerometer_Y_m_s2', 'Accelerometer_Z_m_s2']
gyro_axes = ['Gyroscope_X_deg_s', 'Gyroscope_Y_deg_s', 'Gyroscope_Z_deg_s']

df['Accel_Magnitude'] = np.sqrt(np.square(df[accel_axes]).sum(axis=1))
df['Gyro_Magnitude'] = np.sqrt(np.square(df[gyro_axes]).sum(axis=1))

# Correlation matrix across all continuous features
corr_matrix = df.select_dtypes(include='number').corr()

plt.figure(figsize=(14, 12))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    vmin=-1, vmax=1,
    linewidths=0.5,
    cbar_kws={'label': 'Pearson Correlation'}
)
plt.title('Correlation Matrix of Numeric Features', fontsize=14, pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Analysis 1: Support Vector Machines

In [ ]:
target = 'Shot_Result'
y = df[target]

candidate_features = [
    'Age', 'Height_cm', 'Distance_from_Basket_m', 
    'Shot_Angle_deg', 'Accelerometer_Y_m_s2', 
    'Gyroscope_Y_deg_s', 'Shot_Speed_m_s', 'Release_Angle_deg'
]

# 1. All numeric features
X_set1 = df.select_dtypes(include='number').drop(columns=[target, 'Accel_Magnitude', 'Gyro_Magnitude'])

# 2. All numeric + one-hot encoded Shot_Type
shot_type_dummies = pd.get_dummies(df['Shot_Type'], prefix='Shot_Type', drop_first=True, dtype=float)
X_set2 = pd.concat([X_set1, shot_type_dummies], axis=1)

# 3. Hand-selected candidate features
X_set3 = df[candidate_features].copy()

# 4. Combined sensor magnitudes with remaining numeric features
other_sensor_cols = [c for c in X_set1.columns if c not in accel_axes + gyro_axes]
X_set4 = pd.concat([df[['Accel_Magnitude', 'Gyro_Magnitude']], df[other_sensor_cols]], axis=1)

# Dictionaries to store fitted models and test-set predictions
svm_models = {}
svm_preds = {}
svm_test_targets = {}

### SVM 1 - Only Numeric Variables

In [ ]:
X_tr1, X_te1, y_tr1, y_te1 = train_test_split(X_set1, y, test_size=0.2, random_state=42, stratify=y)

svm1 = make_pipeline(StandardScaler(), SVC())
svm1.fit(X_tr1, y_tr1)

y_pred_svm1 = svm1.predict(X_te1)
svm_models['SVM 1 (All Numeric)'] = svm1
svm_preds['SVM 1 (All Numeric)'] = y_pred_svm1
svm_test_targets['SVM 1 (All Numeric)'] = y_te1

print("=== SVM 1: All Numeric Features ===")
print(f"Accuracy: {accuracy_score(y_te1, y_pred_svm1):.4f}\n")
print(classification_report(y_te1, y_pred_svm1))

### SVM 2 - All Numeric Variables + One-Hot Encoded Shot_Type

In [ ]:
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_set2, y, test_size=0.2, random_state=42, stratify=y)

svm2 = make_pipeline(StandardScaler(), SVC())
svm2.fit(X_tr2, y_tr2)

y_pred_svm2 = svm2.predict(X_te2)
svm_models['SVM 2 (Numeric + Shot_Type)'] = svm2
svm_preds['SVM 2 (Numeric + Shot_Type)'] = y_pred_svm2
svm_test_targets['SVM 2 (Numeric + Shot_Type)'] = y_te2

print("=== SVM 2: Numeric + One-Hot Encoded Shot_Type ===")
print(f"Accuracy: {accuracy_score(y_te2, y_pred_svm2):.4f}\n")
print(classification_report(y_te2, y_pred_svm2))

### SVM 3 - Hand-Selected Variables Only

In [ ]:
X_tr3, X_te3, y_tr3, y_te3 = train_test_split(X_set3, y, test_size=0.2, random_state=42, stratify=y)

svm3 = make_pipeline(StandardScaler(), SVC())
svm3.fit(X_tr3, y_tr3)

y_pred_svm3 = svm3.predict(X_te3)
svm_models['SVM 3 (Hand-Selected)'] = svm3
svm_preds['SVM 3 (Hand-Selected)'] = y_pred_svm3
svm_test_targets['SVM 3 (Hand-Selected)'] = y_te3

print("=== SVM 3: Hand-Selected Features ===")
print(f"Accuracy: {accuracy_score(y_te3, y_pred_svm3):.4f}\n")
print(classification_report(y_te3, y_pred_svm3))

### SVM 4 - Accelerometer and Gyroscope Combined

In [ ]:
X_tr4, X_te4, y_tr4, y_te4 = train_test_split(X_set4, y, test_size=0.2, random_state=42, stratify=y)

svm4 = make_pipeline(StandardScaler(), SVC())
svm4.fit(X_tr4, y_tr4)

y_pred_svm4 = svm4.predict(X_te4)
svm_models['SVM 4 (Sensor Magnitudes)'] = svm4
svm_preds['SVM 4 (Sensor Magnitudes)'] = y_pred_svm4
svm_test_targets['SVM 4 (Sensor Magnitudes)'] = y_te4

print("=== SVM 4: Combined Sensor Magnitudes ===")
print(f"Accuracy: {accuracy_score(y_te4, y_pred_svm4):.4f}\n")
print(classification_report(y_te4, y_pred_svm4))

#### Compare 1st svm to higher poly degree models

In [ ]:
svm_rbf = make_pipeline(StandardScaler(), SVC(kernel="rbf")).fit(
    X_tr1, y_tr1
)
svm_poly = make_pipeline(
    StandardScaler(), SVC(kernel="poly", degree=2)
).fit(X_tr1, y_tr1)

svm_3poly = make_pipeline(
    StandardScaler(), SVC(kernel="poly", degree=3)
).fit(X_tr1, y_tr1)

print("RBF Accuracy:", accuracy_score(y_te1, svm_rbf.predict(X_te1)))
print("Poly 2 Accuracy:", accuracy_score(y_te1, svm_poly.predict(X_te1)))
print("Poly 3 Accuracy:", accuracy_score(y_te1, svm_3poly.predict(X_te1)))

## Analysis 2: Logistic Regression

In [ ]:
# Data structures to store logistic regression models and predictions
lr_models = {}
lr_preds = {}
lr_test_targets = {}

# Fit statsmodels for statistical inference
X_sm1 = sm.add_constant(X_set1.astype(float))
logit_model1 = sm.Logit(y.astype(int), X_sm1).fit()
print("=== Logistic Regression 1 Summary (Inference) ===")
print(logit_model1.summary())

# Fit scikit-learn pipeline for holdout evaluation consistent with SVM
lr1 = make_pipeline(StandardScaler(), LogisticRegression())
lr1.fit(X_tr1, y_tr1)

y_pred_lr1 = lr1.predict(X_te1)
lr_models['LogReg 1 (All Numeric)'] = lr1
lr_preds['LogReg 1 (All Numeric)'] = y_pred_lr1
lr_test_targets['LogReg 1 (All Numeric)'] = y_te1

print("=== Logistic Regression 1 (Holdout Test Evaluation) ===")
print(f"Accuracy: {accuracy_score(y_te1, y_pred_lr1):.4f}\n")
print(classification_report(y_te1, y_pred_lr1))

### Logistic Version #2

In [ ]:
X_sm2 = sm.add_constant(X_set2.astype(float))
logit_model2 = sm.Logit(y.astype(int), X_sm2).fit()
print("=== Logistic Regression 2 Summary (Inference) ===")
print(logit_model2.summary())

lr2 = make_pipeline(StandardScaler(), LogisticRegression())
lr2.fit(X_tr2, y_tr2)

y_pred_lr2 = lr2.predict(X_te2)
lr_models['LogReg 2 (Numeric + Shot_Type)'] = lr2
lr_preds['LogReg 2 (Numeric + Shot_Type)'] = y_pred_lr2
lr_test_targets['LogReg 2 (Numeric + Shot_Type)'] = y_te2

print("=== Logistic Regression 2 (Holdout Test Evaluation) ===")
print(f"Accuracy: {accuracy_score(y_te2, y_pred_lr2):.4f}\n")
print(classification_report(y_te2, y_pred_lr2))

### Logistic Version #3

In [ ]:
X_sm3 = sm.add_constant(X_set3.astype(float))
logit_model3 = sm.Logit(y.astype(int), X_sm3).fit()
print("=== Logistic Regression 3 Summary (Inference) ===")
print(logit_model3.summary())

lr3 = make_pipeline(StandardScaler(), LogisticRegression())
lr3.fit(X_tr3, y_tr3)

y_pred_lr3 = lr3.predict(X_te3)
lr_models['LogReg 3 (Hand-Selected)'] = lr3
lr_preds['LogReg 3 (Hand-Selected)'] = y_pred_lr3
lr_test_targets['LogReg 3 (Hand-Selected)'] = y_te3

print("=== Logistic Regression 3 (Holdout Test Evaluation) ===")
print(f"Accuracy: {accuracy_score(y_te3, y_pred_lr3):.4f}\n")
print(classification_report(y_te3, y_pred_lr3))

### Logistic Version #4

In [ ]:
X_sm4 = sm.add_constant(X_set4.astype(float))
logit_model4 = sm.Logit(y.astype(int), X_sm4).fit()
print("=== Logistic Regression 4 Summary (Inference) ===")
print(logit_model4.summary())

lr4 = make_pipeline(StandardScaler(), LogisticRegression())
lr4.fit(X_tr4, y_tr4)

y_pred_lr4 = lr4.predict(X_te4)
lr_models['LogReg 4 (Sensor Magnitudes)'] = lr4
lr_preds['LogReg 4 (Sensor Magnitudes)'] = y_pred_lr4
lr_test_targets['LogReg 4 (Sensor Magnitudes)'] = y_te4

print("=== Logistic Regression 4 (Holdout Test Evaluation) ===")
print(f"Accuracy: {accuracy_score(y_te4, y_pred_lr4):.4f}\n")
print(classification_report(y_te4, y_pred_lr4))

## Final Results

In [ ]:
results = []

def extract_class_1_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    # Calculate precision, recall, and f1 for class 1 (shot made)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', pos_label=1, zero_division=0
    )
    return acc, precision, recall, f1

for name, y_pred in svm_preds.items():
    y_true = svm_test_targets[name]
    acc, prec, rec, f1 = extract_class_1_metrics(y_true, y_pred)
    results.append({
        'Model Type': 'SVM',
        'Configuration': name.split(': ')[-1],
        'Accuracy': acc,
        'Precision (Class 1)': prec,
        'Recall (Class 1)': rec,
        'F1-Score (Class 1)': f1
    })

for name, y_pred in lr_preds.items():
    y_true = lr_test_targets[name]
    acc, prec, rec, f1 = extract_class_1_metrics(y_true, y_pred)
    results.append({
        'Model Type': 'Logistic Regression',
        'Configuration': name.split(': ')[-1],
        'Accuracy': acc,
        'Precision (Class 1)': prec,
        'Recall (Class 1)': rec,
        'F1-Score (Class 1)': f1
    })

summary_df = pd.DataFrame(results)

# Sort by F1-Score (or Accuracy) as desired
sorted_df = summary_df.sort_values(by='F1-Score (Class 1)', ascending=False).reset_index(drop=True)

metric_cols = ['Accuracy', 'Precision (Class 1)', 'Recall (Class 1)', 'F1-Score (Class 1)']

# Style table: highlight max performer in each metric column with a soft green background
styled_summary = (
    sorted_df.style
    .highlight_max(subset=metric_cols, color='#c6efce')  # Soft pastel green highlight
    .format({col: '{:.4f}' for col in metric_cols})       # Format floats to 4 decimal places
    .set_caption("<b>Model Comparison: Performance on Shot Made (Class 1)</b>")
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '14px'), ('margin-bottom', '8px')]},
        {'selector': 'th', 'props': [('background-color', '#f2f2f2'), ('font-weight', 'bold')]}
    ])
)

print("=== Model Comparison Across Feature Sets (Class 1: Shot Made) ===")
display(styled_summary)

In [ ]:
# Identify top-performing previously calculated models
best_svm_name = max(svm_preds, key=lambda k: accuracy_score(svm_test_targets[k], svm_preds[k]))
best_svm_acc = accuracy_score(svm_test_targets[best_svm_name], svm_preds[best_svm_name])

best_lr_name = max(lr_preds, key=lambda k: accuracy_score(lr_test_targets[k], lr_preds[k]))
best_lr_acc = accuracy_score(lr_test_targets[best_lr_name], lr_preds[best_lr_name])

print(f"Top SVM Model: {best_svm_name} (Accuracy: {best_svm_acc:.4f})")
print(classification_report(svm_test_targets[best_svm_name], svm_preds[best_svm_name]))

print(f"Top Logistic Regression Model: {best_lr_name} (Accuracy: {best_lr_acc:.4f})")
print(classification_report(lr_test_targets[best_lr_name], lr_preds[best_lr_name]))

# Comparative Bar Chart & Confusion Matrices using previously calculated results
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Model Comparison Bar Chart
configurations = list(summary_pivot.index)
x_pos = np.arange(len(configurations))
width = 0.35

svm_accs = [summary_pivot.loc[c, 'SVM'] for c in configurations]
lr_accs = [summary_pivot.loc[c, 'Logistic Regression'] for c in configurations]

axes[0].bar(x_pos - width/2, svm_accs, width, label='SVM', color='steelblue')
axes[0].bar(x_pos + width/2, lr_accs, width, label='Logistic Regression', color='coral')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Test Accuracy by Model and Feature Set')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(configurations, rotation=25, ha='right')
axes[0].set_ylim(0, 1.0)
axes[0].legend()

# Confusion Matrix for Best SVM
ConfusionMatrixDisplay.from_predictions(
    svm_test_targets[best_svm_name], svm_preds[best_svm_name], 
    ax=axes[1], cmap='Blues', colorbar=False
)
axes[1].set_title(f'Best SVM: {best_svm_name}')

# Confusion Matrix for Best Logistic Regression
ConfusionMatrixDisplay.from_predictions(
    lr_test_targets[best_lr_name], lr_preds[best_lr_name], 
    ax=axes[2], cmap='Oranges', colorbar=False
)
axes[2].set_title(f'Best Logistic: {best_lr_name}')

plt.tight_layout()
plt.show()